In [5]:
import pandas as pd

# ✅ Show ALL rows (no truncation)
pd.set_option('display.max_rows', None)

# ✅ Show ALL columns (fully visible)
pd.set_option('display.max_columns', None)

# ✅ Prevent cutting off long strings (full metadata, filenames, tags)
pd.set_option('display.max_colwidth', None)

# ✅ Wider output — less ugly wrapping
pd.set_option('display.width', 2000)


In [2]:
# -----######-----###### MAIN IMPORTS -----######-----######
import re, json, requests, pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

# ---------- TQM ----------
def _tqm_bar(step, total, label):
    width = 28
    frac = 0 if total == 0 else step/float(total)
    filled = int(width*frac)
    bar = "█"*filled + " "*(width-filled)
    print(f"TQM | {label}: {int(frac*100):3d}%|{bar}| {step}/{total}")

# ==============================
#  _bandcamp_2710_album_GET_df_tracks
# ==============================
# -----######-----###### CORE IMPORTABLE FUNCTION -----######-----######
def _bandcamp_2710_album_GET_df_tracks(
    urls_or_df,
    url_col=None,
    purchase_date=None,          # "YYYY-MM-DD" — applied to all rows
    default_key=None,            # e.g., "8A"
    default_bpm=None,            # e.g., "126"
    default_genre=None           # e.g., "Detroit House"
):
    """
    Scrape Bandcamp album OR single-track pages and return track-level metadata + a rename key column.

    Columns:
      url, album_name, artist, label, release_date, release_year, release_month, release_day,
      track_number, track_title, is_remix, remixer, mix_type, genre, bpm, key, rename_key
    """

    # ---------- helpers ----------
    def _clean_date(dtxt):
        dtxt = (dtxt or "").strip()
        for fmt in ["%B %d %Y", "%B %d, %Y", "%d %B %Y", "%Y-%m-%d"]:
            try:
                return datetime.strptime(dtxt.replace(",", ""), fmt).strftime("%Y-%m-%d")
            except Exception:
                pass
        return dtxt or None

    def _extract_release_parts(date_str):
        if not date_str:
            return None, None, None, None
        m = re.search(r"(\d{4})-(\d{2})-(\d{2})", date_str or "")
        if m:
            y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
            return date_str, y, mo, d
        y = re.search(r"(\d{4})", date_str or "")
        return date_str, int(y.group(1)) if y else None, None, None

    def _parse_purchase_parts(pdate):
        if not pdate:
            return None, None, None
        try:
            dt = datetime.strptime(str(pdate), "%Y-%m-%d")
            return dt.year, dt.month, dt.day
        except Exception:
            m = re.search(r"(\d{4})[\-\/\.](\d{1,2})[\-\/\.](\d{1,2})", str(pdate))
            if m:
                return int(m.group(1)), int(m.group(2)), int(m.group(3))
            return None, None, None

    # ---- FIXED mix/remixer parser ----
    _STD_TYPES = [
        "extended mix","club mix","club tool","radio edit","instrumental",
        "dub","edit","version","vip","bootleg"
    ]
    def _parse_mix_info(title):
        """
        Rules:
         - "(Someone Remix)" or " - Someone Remix" -> remixer=Someone, mix_type='Remix', is_remix=True
         - "(Extended Mix / Club Tool / Dub / Edit / Version / VIP / Bootleg / Radio Edit / Instrumental)"
               -> mix_type=TitleCase(inner), NO remixer, is_remix=False
         - "(... mix)" where inner endswith ' mix' (not 'remix') -> NO remixer, mix_type='Original Mix', is_remix=False
         - Otherwise -> NO remixer, mix_type=None (later forced to 'Original Mix'), is_remix=False
        """
        if not title:
            return False, None, None

        t = title.strip()

        # Pattern A: trailing dash ' - Someone Remix'
        m_dash = re.search(r"[-–]\s*(?P<who>.+?)\s+Remix$", t, flags=re.IGNORECASE)
        if m_dash:
            return True, m_dash.group("who").strip(), "Remix"

        # Pattern B: parenthetical at end
        paren = re.search(r"\(([^)]{2,})\)$", t)
        if paren:
            inner = paren.group(1).strip()
            inner_low = inner.lower()

            # B1) classic remix
            if inner_low.endswith("remix"):
                who = inner[:-5].strip()  # drop 'remix'
                who = who[:-1].strip() if who.endswith("-") else who  # trim trailing dash if any
                return True, (who if who else None), "Remix"

            # B2) standard known types (no remixer)
            if inner_low in _STD_TYPES:
                return False, None, inner.title()

            # B3) anything that ends with ' mix' (not 'remix') => Original Mix, no remixer
            if inner_low.endswith(" mix"):
                return False, None, "Original Mix"

            # otherwise nothing we trust
            return False, None, None

        # Default: nothing found
        return False, None, None

    def _parse_tralbum_json(soup):
        for sc in soup.find_all("script"):
            if sc.string and "TralbumData" in sc.string:
                m = re.search(r"TralbumData\s*=\s*(\{.*?\});", sc.string, flags=re.DOTALL)
                if m:
                    try:
                        return json.loads(m.group(1))
                    except Exception:
                        pass
        return None

    def _parse_embed_json(soup):
        for sc in soup.find_all("script"):
            if sc.string and "EmbedData" in sc.string:
                m = re.search(r"EmbedData\s*=\s*(\{.*?\});", sc.string, flags=re.DOTALL)
                if m:
                    try:
                        return json.loads(m.group(1))
                    except Exception:
                        pass
        return None

    def _guess_label(soup):
        for a in soup.select("a[href*='/label/']"):
            txt = (a.get_text() or "").strip()
            if txt:
                return txt
        return None

    def _get_artist(soup, tralbum):
        # Prefer JSON artist when available
        if tralbum and tralbum.get("artist"):
            at = (tralbum.get("artist") or "").strip()
            if at:
                return at
        cand = ["span[itemprop='byArtist'] a", ".artist-override"]
        for sel in cand:
            el = soup.select_one(sel)
            if el:
                txt = (el.get_text() or "").strip()
                if txt:
                    return txt
        m = soup.find("meta", property="og:site_name")
        if m and m.get("content"):
            return m["content"].strip()
        return None

    def _get_album_title(soup, tralbum, embed):
        # album pages -> album title; track pages often have current.title == track title; fallback sensibly
        if tralbum and isinstance(tralbum.get("current"), dict):
            cur = tralbum["current"]
            if cur.get("title"):
                return cur["title"].strip()

        # EmbedData sometimes has album_title (varies)
        if embed and embed.get("album_title"):
            at = (embed["album_title"] or "").strip()
            if at:
                return at

        el = soup.select_one("h2.trackTitle")
        if el:
            return el.get_text(strip=True)

        if soup.title and soup.title.string:
            return soup.title.string.strip()

        return None

    def _get_release_text(soup):
        credits_div = soup.find("div", class_="tralbumData tralbum-credits")
        if credits_div:
            txt = credits_div.get_text(" ", strip=True)
            m = re.search(r"released\s+([A-Za-z]+\s+\d{1,2},\s+\d{4})", txt, flags=re.IGNORECASE)
            if m:
                return m.group(1)
            m2 = re.search(r"([A-Za-z]+\s+\d{1,2},\s+\d{4})", txt)
            if m2:
                return m2.group(1)
        return None

    def _tracks_from_tralbum(tralbum, soup):
        """
        Works for albums and single tracks:
        - If 'trackinfo' list exists & non-empty -> use it (album or track page)
        - Else -> attempt to pull a single title from DOM/meta
        """
        tracks = []
        ti = None
        if tralbum and isinstance(tralbum.get("trackinfo"), list):
            ti = tralbum["trackinfo"]

        if ti and len(ti) > 0:
            for i, t in enumerate(ti, start=1):
                title = t.get("title")
                tracks.append({"track_number": i, "track_title": title})
            return tracks

        # Single track fallback (DOM)
        # Try Bandcamp track page patterns
        single_candidates = [
            "h2.trackTitle",
            "h3#track-name",
            "meta[property='og:title']"
        ]
        for sel in single_candidates:
            el = soup.select_one(sel)
            if el:
                if el.name == "meta":
                    title = el.get("content", "").strip()
                else:
                    title = el.get_text(strip=True)
                if title:
                    return [{"track_number": 1, "track_title": title}]
        return tracks

    def _detect_genre(tralbum, embed, default_genre):
        # Prefer tags from embedded JSON; fallback to provided default
        tag_lists = []
        for source in [tralbum, embed]:
            if not source:
                continue
            if isinstance(source.get("tags"), list):
                tag_lists.append(source["tags"])
            cur = source.get("current") if isinstance(source.get("current"), dict) else None
            if cur and isinstance(cur.get("tags"), list):
                tag_lists.append(cur["tags"])
        for tags in tag_lists:
            for tag in tags:
                tag = (tag or "").strip()
                if tag:
                    return tag
        return default_genre or ""

    def _scrape_one(url):
        r = requests.get(url)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        tralbum = _parse_tralbum_json(soup)
        embed   = _parse_embed_json(soup)

        album_name = _get_album_title(soup, tralbum, embed)
        artist     = _get_artist(soup, tralbum)

        rel_txt = _get_release_text(soup)
        rel_str = _clean_date(rel_txt)
        release_date, release_year, release_month, release_day = _extract_release_parts(rel_str or rel_txt)

        label = _guess_label(soup) or "Self Release"

        # Tracks (album or single)
        tracks = _tracks_from_tralbum(tralbum, soup)

        # Genre detection (prefer page tags; fallback to default_genre)
        genre_value = _detect_genre(tralbum, embed, default_genre)

        # Prepare rename key constants
        ky  = default_key if default_key is not None else ""
        bpm = default_bpm if default_bpm is not None else ""
        py, pmo, pd = _parse_purchase_parts(purchase_date)

        rows = []
        for t in tracks:
            tn = t.get("track_number")
            title = (t.get("track_title") or "").strip()

            is_remix, remixer, mix_type = _parse_mix_info(title)

            # Force Original Mix if nothing parsed
            if not mix_type:
                mix_type = "Original Mix"
                is_remix = False
                remixer  = None

            rename_key = (
                f"TRkw_{title}"
                f"_ARkw_{artist if artist else ''}"
                f"_MXkw_{mix_type if mix_type else ''}"
                f"_KYkw_{ky}"
                f"_BPkw_{bpm}"
                f"_GNkw_{genre_value}"
                f"_RMkw_{remixer if remixer else ''}"
                f"_LBkw_{label if label else ''}"
                f"_RYkw_{release_year if release_year is not None else ''}"
                f"_{release_month if release_month is not None else ''}"
                f"_{release_day if release_day is not None else ''}"
                f"_PYkw_{py if py is not None else ''}"
                f"_{pmo if pmo is not None else ''}"
                f"_{pd if pd is not None else ''}"
            )

            rows.append({
                "url": url,
                "album_name": album_name,
                "artist": artist,
                "label": label,
                "release_date": release_date,
                "release_year": release_year,
                "release_month": release_month,
                "release_day": release_day,
                "track_number": tn,
                "track_title": title,
                "is_remix": is_remix,
                "remixer": remixer,
                "mix_type": mix_type,
                "genre": genre_value,
                "bpm": bpm,
                "key": ky,
                "rename_key": rename_key
            })
        return rows

    # ---------- dispatch ----------
    if isinstance(urls_or_df, pd.DataFrame):
        urls = urls_or_df[url_col].dropna().astype(str).tolist()
        out_rows, total = [], len(urls)
        for i, u in enumerate(urls, start=1):
            _tqm_bar(i, total, "Bandcamp scrape")
            try:
                out_rows.extend(_scrape_one(u))
            except Exception as e:
                out_rows.append({
                    "url": u, "album_name": None, "artist": None, "label": "Self Release",
                    "release_date": None, "release_year": None, "release_month": None, "release_day": None,
                    "track_number": None, "track_title": None,
                    "is_remix": False, "remixer": None, "mix_type": "Original Mix",
                    "genre": (default_genre or ""), "bpm": (default_bpm or ""), "key": (default_key or ""),
                    "rename_key": None, "_error": str(e)
                })
        return pd.DataFrame(out_rows)

    elif isinstance(urls_or_df, str):
        _tqm_bar(1, 1, "Bandcamp scrape")
        return pd.DataFrame(_scrape_one(urls_or_df))

    else:
        urls = list(urls_or_df)
        out_rows, total = [], len(urls)
        for i, u in enumerate(urls, start=1):
            _tqm_bar(i, total, "Bandcamp scrape")
            out_rows.extend(_scrape_one(u))
        return pd.DataFrame(out_rows)


In [23]:
# Example (single URL):
df_bc = _bandcamp_2710_album_GET_df_tracks(
    "https://brvss.bandcamp.com/track/la-beb-brvss-calor-n-mix",
    purchase_date="2025-10-27",
    default_genre="Rgtton_Gtech_Latin"   # optional; page tags override if present
)
df_bc[["track_number","track_title","is_remix","remixer","mix_type","genre","rename_key"]]

# Example (DataFrame column):
# df_bc = _bandcamp_2710_album_GET_df_tracks(df, url_col="bandcamp_url", purchase_date="2025-10-27")
df_bc

TQM | Bandcamp scrape: 100%|████████████████████████████| 1/1


,url,album_name,artist,label,release_date,release_year,release_month,release_day,track_number,track_title,is_remix,remixer,mix_type,genre,bpm,key,rename_key
0,https://brvss.bandcamp.com/track/la-beb-brvss-calor-n-mix,la bebé (brvss calorón mix),brvss,Self Release,2025-09-04,2025,9,4,1,la bebé (brvss calorón mix),False,None,Original Mix,Rgtton_Gtech_Latin,,,TRkw_la bebé (brvss calorón mix)_ARkw_brvss_MXkw_Original Mix_KYkw__BPkw__GNkw_Rgtton_Gtech_Latin_RMkw__LBkw_Self Release_RYkw_2025_9_4_PYkw_2025_10_27


In [12]:
# df_bc

# TRkw_{track_name}_
# ARkw_{artists}_
# MXkw_{mix_name}_
# KYkw_{key}_
# BPkw_{bpm}_
# GNkw_{genre}_
# RMkw_{remixers}_
# LBkw_{label}_
# RYkw_{release_year}_{release_month}_{release_day}_
# PYkw_{purchase_year}_{purchase_month}_{purchase_day}

# TRkw_la bebé (brvss jungleton mix)_
# ARkw_brvss_
# MXkw__
# KYkw__
# BPkw__
# GNkw__
# RMkw__
# LBkw_Self Release_
# RYkw_2025_9_29_
# PYkw_2025_10_27

## MATCH with folder !

In [5]:
# ==============================
#  TQM helper (prints to stdout)
# ==============================
def _tqm_bar(step, total, label):
    width = 28
    frac = 0 if total == 0 else step / float(total)
    filled = int(width * frac)
    bar = "█" * filled + " " * (width - filled)
    print(f"TQM | {label}: {int(frac*100):3d}%|{bar}| {step}/{total}")

# ==============================
#  Text + filename utils
# ==============================
import os, re, unicodedata
from difflib import SequenceMatcher

def _strip_accents(s):
    if not isinstance(s, str):
        return ""
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def _normalize_text(s):
    if s is None: s = ""
    s = _strip_accents(s).lower()
    s = re.sub(r"[\(\[\{].*?[\)\]\}]", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _tokenize_core(s):
    toks = _normalize_text(s).split()
    stop = {
        "the","a","an","and","ft","feat","featuring","vs","with",
        "mix","remix","edit","version","dub","tool","club","original",
        "extended","radio","vip","rework","bootleg","instrumental","acapella"
    }
    keep_short = {"la","el","de","del","al","yo","tu","mi","ya"}
    out = []
    for t in toks:
        if (len(t) >= 3 or t in keep_short) and t not in stop:
            out.append(t)
    return out

def _token_set(s):
    return set(_tokenize_core(s))

def _seq_ratio(a, b):
    try:
        return SequenceMatcher(None, a, b).ratio()
    except Exception:
        return 0.0

def _jaccard(a_tokens, b_tokens):
    if not a_tokens or not b_tokens:
        return 0.0
    inter = len(a_tokens & b_tokens)
    union = len(a_tokens | b_tokens)
    return 0.0 if union == 0 else inter / union

def _score_title_vs_stem(title, file_norm, file_tokens):
    title_norm = _normalize_text(title)
    title_tokens = _token_set(title)
    s = _seq_ratio(title_norm, file_norm)   # 0..1
    j = _jaccard(title_tokens, file_tokens) # 0..1
    return 0.6*s + 0.4*j

def _sanitize_filename(name):
    """
    Convert any string (e.g., rename_key) into a filesystem-safe filename stem.
    Keeps unicode letters/digits, replaces spaces with single spaces, trims.
    """
    if name is None: name = ""
    name = name.strip()
    # kill path separators
    name = name.replace("/", "-").replace("\\", "-")
    # collapse whitespace
    name = re.sub(r"\s+", " ", name)
    # optional hard cleanup of other forbidden chars on macOS
    name = re.sub(r'[:*?"<>|]', "-", name)
    return name.strip()

def _safe_unique_path(dirpath, stem, ext):
    """
    If dirpath/stem.ext exists, append _dupN.
    """
    target = os.path.join(dirpath, f"{stem}{ext}")
    if not os.path.exists(target):
        return target
    n = 1
    while True:
        p = os.path.join(dirpath, f"{stem}_dup{n}{ext}")
        if not os.path.exists(p):
            return p
        n += 1

# ==============================
#  Index audio files (recursive)
# ==============================
def _index_audio_files_audioonly(base_dir, audio_extensions):
    base_dir = os.path.abspath(base_dir)
    files = []
    for root, dirs, fnames in os.walk(base_dir):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        for f in fnames:
            if f.startswith("._") or f == ".DS_Store":
                continue
            ext = os.path.splitext(f)[1].lower().lstrip(".")
            if audio_extensions and ext not in audio_extensions:
                continue
            full = os.path.join(root, f)
            stem = os.path.splitext(f)[0]
            files.append((full, f, stem, _normalize_text(stem), _token_set(stem)))
    return files

# ==============================
#  Hungarian (square) + rectangular padding
# ==============================
def _hungarian_min_cost(cost):
    n = len(cost)
    u = [0.0]*(n+1); v = [0.0]*(n+1); p = [0]*(n+1); way = [0]*(n+1)
    for i in range(1, n+1):
        p[0] = i; j0 = 0
        minv = [float("inf")]*(n+1); used = [False]*(n+1)
        while True:
            used[j0] = True
            i0 = p[j0]; delta = float("inf"); j1 = 0
            for j in range(1, n+1):
                if not used[j]:
                    cur = cost[i0-1][j-1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j] = cur; way[j] = j0
                    if minv[j] < delta:
                        delta = minv[j]; j1 = j
            for j in range(0, n+1):
                if used[j]:
                    u[p[j]] += delta; v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while True:
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
            if j0 == 0: break
    assignment = [0]*n
    for j in range(1, n+1):
        assignment[p[j]-1] = j-1
    return assignment  # row i -> col j

# -----######-----######  CORE IMPORTABLE FUNCTION  -----######-----######
def _match_2710_rect_GET_df_paths(
    df,
    base_dir,
    audio_extensions=None,   # e.g. {"mp3","wav","aiff","aif"}
    min_warn_score=0.62
):
    """
    Rectangular-safe global matcher (audio-only). Adds match_* columns.
    """
    import pandas as pd
    if 'track_title' not in df.columns:
        raise ValueError("DataFrame must have a 'track_title' column.")
    if audio_extensions is None or len(audio_extensions) == 0:
        audio_extensions = {"mp3","wav","aiff","aif"}

    files = _index_audio_files_audioonly(base_dir, audio_extensions)
    n_rows = len(df); n_files = len(files)

    # score matrix (rows x files) in [0,1]
    _tqm_bar(0, n_rows, "Scoring Matrix")
    scores = []
    for i, row in enumerate(df.itertuples(index=False)):
        title = getattr(row, 'track_title', '')
        row_scores = []
        for (path, fname, stem, stem_norm, tokens) in files:
            row_scores.append(_score_title_vs_stem(title, stem_norm, tokens))
        scores.append(row_scores)
        _tqm_bar(i+1, n_rows, "Scoring Matrix")

    # pad to square
    N = max(n_rows, n_files)
    cost = [[1.0]*N for _ in range(N)]
    for i in range(n_rows):
        for j in range(n_files):
            s = scores[i][j]
            s = 0.0 if s < 0 else (1.0 if s > 1.0 else s)
            cost[i][j] = 1.0 - s

    assignment = _hungarian_min_cost(cost)

    # attach
    df = df.copy()
    df['match_path'] = None
    df['match_filename'] = None
    df['match_score'] = 0.0
    df['match_method'] = 'hungarian-rect'
    df['low_confidence'] = False
    df['unmatched_reason'] = ""

    _tqm_bar(0, n_rows, "Assigning")
    for i, j in enumerate(assignment[:n_rows]):
        if j >= n_files:
            df.at[i, 'unmatched_reason'] = 'no_file'
            df.at[i, 'low_confidence'] = True
            _tqm_bar(i+1, n_rows, "Assigning"); continue
        path, fname, stem, stem_norm, tokens = files[j]
        sc = float(1.0 - cost[i][j])
        df.at[i, 'match_path'] = path
        df.at[i, 'match_filename'] = fname
        df.at[i, 'match_score'] = round(sc, 4)
        df.at[i, 'low_confidence'] = bool(sc < min_warn_score)
        _tqm_bar(i+1, n_rows, "Assigning")

    return df, files, scores  # return files/scores for interactive step

# -----######-----######  CORE IMPORTABLE FUNCTION  -----######-----######
def _confirm_2710_pairs_GET_df_renamed(
    df,
    base_dir,
    files,
    scores,
    dry_run=False
):
    """
    Interactive: for each row, propose the closest file → ask to confirm.
    If 'y' → rename file to df.rename_key (sanitized) + same extension, in-place.
    If 'n' → show next best 3 candidates; user can pick 1/2/3 or 's' to skip.

    Returns df with columns:
      old_path, new_path, renamed, user_choice, rename_error
    """
    import pandas as pd

    if 'rename_key' not in df.columns:
        raise ValueError("DataFrame must have a 'rename_key' column.")

    df = df.copy()
    df['old_path'] = df.get('match_path', None)
    df['new_path'] = None
    df['renamed'] = False
    df['user_choice'] = ""
    df['rename_error'] = ""

    n = len(df)
    _tqm_bar(0, n, "Confirm & Rename")

    # helper: rank top-k candidates for a row
    def topk_for_row(i, k=4):
        row_scores = scores[i]
        ranked = sorted(
            [(j, row_scores[j]) for j in range(len(files))],
            key=lambda x: x[1],
            reverse=True
        )
        return ranked[:k]

    for i, row in df.iterrows():
        title = row.get('track_title', '')
        rk = row.get('rename_key', '')
        ext_current = ""
        suggested_idx = None

        # suggested = best by scores
        top = topk_for_row(i, k=4)
        if top:
            suggested_idx = top[0][0]
            sug_path, sug_fname, sug_stem, _, _ = files[suggested_idx]
            ext_current = os.path.splitext(sug_fname)[1]
        else:
            df.at[i,'rename_error'] = "no_candidates"
            _tqm_bar(i+1, n, "Confirm & Rename")
            continue

        print("\n" + "-"*66)
        print(f"[{i+1}/{n}]  TRACK: {title}")
        print(f"  ▶ Proposed file: {sug_fname}")
        print(f"  ▶ Score: {scores[i][suggested_idx]:.3f}")
        ans = input("Accept? (y = yes / n = no / s = skip) ").strip().lower()

        if ans == "y":
            df.at[i,'user_choice'] = "y"
            dirpath = os.path.dirname(files[suggested_idx][0])
            new_stem = _sanitize_filename(rk)
            new_path = _safe_unique_path(dirpath, new_stem, ext_current)
            if dry_run:
                df.at[i,'new_path'] = new_path
                df.at[i,'renamed'] = False
            else:
                try:
                    os.rename(files[suggested_idx][0], new_path)
                    df.at[i,'new_path'] = new_path
                    df.at[i,'renamed'] = True
                    # update in-memory file index so it won’t get re-picked later
                    files[suggested_idx] = (new_path, os.path.basename(new_path),
                                            os.path.splitext(os.path.basename(new_path))[0],
                                            _normalize_text(os.path.splitext(os.path.basename(new_path))[0]),
                                            _token_set(os.path.splitext(os.path.basename(new_path))[0]))
                except Exception as e:
                    df.at[i,'rename_error'] = f"{type(e).__name__}: {e}"
            _tqm_bar(i+1, n, "Confirm & Rename")
            continue

        if ans == "s":
            df.at[i,'user_choice'] = "s"
            _tqm_bar(i+1, n, "Confirm & Rename")
            continue

        # 'n' → present alternatives
        df.at[i,'user_choice'] = "n"
        print("Alternatives:")
        alts = top[1:] if len(top) > 1 else []
        if not alts:
            print("  (no alternatives)")
            _tqm_bar(i+1, n, "Confirm & Rename")
            continue

        for idx, (j, sc) in enumerate(alts, start=1):
            print(f"  {idx}) {files[j][1]}   (score={sc:.3f})")
        pick = input("Pick 1/2/3 or 's' to skip: ").strip().lower()

        if pick in {"1","2","3"}:
            pick_i = int(pick)-1
            if pick_i < len(alts):
                chosen_j = alts[pick_i][0]
                dirpath = os.path.dirname(files[chosen_j][0])
                ext = os.path.splitext(files[chosen_j][1])[1]
                new_stem = _sanitize_filename(rk)
                new_path = _safe_unique_path(dirpath, new_stem, ext)
                if dry_run:
                    df.at[i,'new_path'] = new_path
                    df.at[i,'renamed'] = False
                else:
                    try:
                        os.rename(files[chosen_j][0], new_path)
                        df.at[i,'new_path'] = new_path
                        df.at[i,'renamed'] = True
                        files[chosen_j] = (new_path, os.path.basename(new_path),
                                           os.path.splitext(os.path.basename(new_path))[0],
                                           _normalize_text(os.path.splitext(os.path.basename(new_path))[0]),
                                           _token_set(os.path.splitext(os.path.basename(new_path))[0]))
                    except Exception as e:
                        df.at[i,'rename_error'] = f"{type(e).__name__}: {e}"
        else:
            # skip
            pass

        _tqm_bar(i+1, n, "Confirm & Rename")

    return df


In [6]:
# External parameters (you control these)
base_dir = "/Users/yerik/Downloads/_brvss"   # <- change to your root folder
# ===== External params you control =====
#base_dir = "/Volumes/HD_back_UP/ALL_AU_DEP"   # <- point to the album's folder
audio_extensions = {"aiff","wav","flac","mp3","m4a","alac","aif"}  # set() to allow all
min_warn_score = 0.62  # flag anything below this as low_confidence

# ===== Run global 1-to-1 matching =====
df = _match_2710_elim_GET_df_paths(
    df=df_bc,
    base_dir=base_dir,
    audio_extensions=audio_extensions,
    min_warn_score=min_warn_score
)

# ===== Quick QA =====
df[["track_title","match_filename","match_score","low_confidence"]]


ValueError: Row/File count mismatch: rows=4, files=5. Elimination approach requires equality.

In [14]:
df_bc

,url,album_name,artist,label,release_date,release_year,release_month,release_day,track_number,track_title,is_remix,remixer,mix_type,genre,bpm,key,rename_key
0,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,9,29,1,la bebé (brvss jungleton mix),False,None,Original Mix,Ragaeton,,,TRkw_la bebé (brvss jungleton mix)_ARkw_brvss_MXkw_Original Mix_KYkw__BPkw__GNkw_Ragaeton_RMkw__LBkw_Self Release_RYkw_2025_9_29_PYkw_2025_10_27
1,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,9,29,2,mes tas tentando (brvss club tool),False,None,Original Mix,Ragaeton,,,TRkw_mes tas tentando (brvss club tool)_ARkw_brvss_MXkw_Original Mix_KYkw__BPkw__GNkw_Ragaeton_RMkw__LBkw_Self Release_RYkw_2025_9_29_PYkw_2025_10_27
2,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,9,29,3,rather lie (y2kid's eurodance mix),False,None,Original Mix,Ragaeton,,,TRkw_rather lie (y2kid's eurodance mix)_ARkw_brvss_MXkw_Original Mix_KYkw__BPkw__GNkw_Ragaeton_RMkw__LBkw_Self Release_RYkw_2025_9_29_PYkw_2025_10_27
3,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,9,29,4,soy mexicano (Pearl Jetta remix),True,Pearl Jetta,Remix,Ragaeton,,,TRkw_soy mexicano (Pearl Jetta remix)_ARkw_brvss_MXkw_Remix_KYkw__BPkw__GNkw_Ragaeton_RMkw_Pearl Jetta_LBkw_Self Release_RYkw_2025_9_29_PYkw_2025_10_27


In [5]:
# -----######-----###### MAIN IMPORTS -----######-----######
import re, json, requests, pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

# ---------- TQM ----------
def _tqm_bar(step, total, label):
    width = 28
    frac = 0 if total == 0 else step/float(total)
    filled = int(width*frac)
    bar = "█"*filled + " "*(width-filled)
    print(f"TQM | {label}: {int(frac*100):3d}%|{bar}| {step}/{total}")

# ==============================
#  _bandcamp_2710_album_GET_df_tracks
# ==============================
# -----######-----###### CORE IMPORTABLE FUNCTION -----######-----######
def _bandcamp_2710_album_GET_df_tracks(
    urls_or_df,
    url_col=None
):
    """
    Scrape Bandcamp album/single pages and return track-level metadata.

    Parameters
    ----------
    urls_or_df : str | list[str] | pandas.DataFrame
        - A single Bandcamp URL string, or
        - A list of Bandcamp URLs, or
        - A DataFrame; if provided, also pass `url_col` with the column name holding URLs.
    url_col : str or None
        Required only when urls_or_df is a DataFrame. The column that contains Bandcamp URLs.

    Returns
    -------
    pandas.DataFrame
        Columns:
          - url
          - album_name
          - artist
          - label
          - release_date      (YYYY-MM-DD when possible, otherwise raw text)
          - release_year
          - track_number
          - track_title
          - is_remix          (True/False)
          - remixer           (if parsed)
          - mix_type          (if parsed)
    """
    # ---------- helpers ----------
    def _clean_date(dtxt):
        # Try parsing common patterns like "September 29, 2025"
        dtxt = (dtxt or "").strip()
        for fmt in ["%B %d %Y", "%B %d, %Y", "%d %B %Y", "%Y-%m-%d"]:
            try:
                return datetime.strptime(dtxt.replace(",", ""), fmt).strftime("%Y-%m-%d")
            except Exception:
                pass
        return dtxt or None

    def _extract_release_year(dtxt):
        if not dtxt:
            return None
        m = re.search(r"(\d{4})", dtxt)
        return int(m.group(1)) if m else None

    def _parse_mix_info(title):
        """
        Extract remixer and mix type from typical patterns:
          "Track (Remixer Remix)", "Track (Extended Mix)", "Track (VIP)", "Track (XYZ Edit)", "Track (Dub)"
        """
        if not title:
            return False, None, None

        t = title.strip()
        low = t.lower()
        # quick boolean for remix-like words
        remix_markers = ["remix", "refix", "rework", "edit", "dub", "vip", "version", "extended mix",
                         "radio edit", "instrumental", "club mix", "bootleg"]
        is_remix = any(w in low for w in remix_markers)

        remixer, mix_type = None, None
        # look inside parentheses
        paren = re.search(r"\(([^)]{2,})\)$", t)
        if paren:
            inner = paren.group(1).strip()
            # Try specific "(Something Remix)"
            m = re.search(r"^(?P<who>.+?)\s+(?P<mix>(?:re-?mix|remix|refix|rework|edit|dub|vip|version|extended mix|radio edit|instrumental|club mix|bootleg))$",
                          inner, flags=re.IGNORECASE)
            if m:
                remixer = m.group("who").strip()
                mix_type = m.group("mix").title()
            else:
                # Or pure mix type e.g., "(Extended Mix)"
                m2 = re.search(r"^(?P<mix>(?:re-?mix|remix|refix|rework|edit|dub|vip|version|extended mix|radio edit|instrumental|club mix|bootleg))$",
                               inner, flags=re.IGNORECASE)
                if m2:
                    mix_type = m2.group("mix").title()

        # If still nothing, try simple " - XYZ Remix" at end
        if not remixer and not mix_type:
            m3 = re.search(r"[-–]\s*(?P<who>.+?)\s+(?P<mix>Remix)$", t, flags=re.IGNORECASE)
            if m3:
                remixer = m3.group("who").strip()
                mix_type = "Remix"
                is_remix = True

        return bool(is_remix), remixer, mix_type

    def _parse_tralbum_json(soup):
        """
        Bandcamp embeds a JS object 'TralbumData' containing trackinfo, album title, etc.
        """
        for sc in soup.find_all("script"):
            if sc.string and "TralbumData" in sc.string:
                # Extract JSON after 'TralbumData = ' up to trailing ';'
                m = re.search(r"TralbumData\s*=\s*(\{.*?\});", sc.string, flags=re.DOTALL)
                if m:
                    try:
                        return json.loads(m.group(1))
                    except Exception:
                        pass
        return None

    def _guess_label(soup):
        # Try common places for a label link, fallback to None
        # Bandcamp often uses a sidebar "Label" link on label pages; artist pages may not have one.
        # We'll try og:site_name/artist first; if not, return None here; the caller will default to "Self Release".
        # Sometimes a label appears as an anchor containing '/label/'.
        for a in soup.select("a[href*='/label/']"):
            txt = (a.get_text() or "").strip()
            if txt:
                return txt
        return None

    def _get_artist(soup):
        # Common selectors
        cand = [
            "span[itemprop='byArtist'] a",
            ".artist-override",
            "a[href*='bandcamp.com'] .title",  # fallback
        ]
        for sel in cand:
            el = soup.select_one(sel)
            if el:
                txt = (el.get_text() or "").strip()
                if txt:
                    return txt
        # Fallback to meta tag
        m = soup.find("meta", property="og:site_name")
        if m and m.get("content"):
            return m["content"].strip()
        return None

    def _get_album_title(soup, tralbum):
        if tralbum and tralbum.get("current") and tralbum["current"].get("title"):
            return tralbum["current"]["title"].strip()
        el = soup.select_one("h2.trackTitle")
        if el:
            return el.get_text(strip=True)
        # Fallback to page title
        if soup.title and soup.title.string:
            return soup.title.string.strip()
        return None

    def _get_release_text(soup):
        # The release text often appears inside credits div
        credits_div = soup.find("div", class_="tralbumData tralbum-credits")
        if credits_div:
            txt = credits_div.get_text(" ", strip=True)
            # find after 'released'
            m = re.search(r"released\s+([A-Za-z]+\s+\d{1,2},\s+\d{4})", txt, flags=re.IGNORECASE)
            if m:
                return m.group(1)
            # fallback: any Month Day Year
            m2 = re.search(r"([A-Za-z]+\s+\d{1,2},\s+\d{4})", txt)
            if m2:
                return m2.group(1)
        return None

    def _tracks_from_tralbum(tralbum):
        tracks = []
        if tralbum and tralbum.get("trackinfo"):
            for i, t in enumerate(tralbum["trackinfo"], start=1):
                title = t.get("title")
                tracks.append({"track_number": i, "track_title": title})
        return tracks

    def _tracks_from_dom(soup):
        tracks = []
        rows = soup.select("table#track_table tr.track_row_view")
        if rows:
            for i, r in enumerate(rows, start=1):
                tt = r.select_one("span.track-title")
                title = tt.get_text(strip=True) if tt else None
                tracks.append({"track_number": i, "track_title": title})
        else:
            # Sometimes on singles the DOM structure is simpler
            title_el = soup.select_one("h3#track-name")
            if title_el:
                tracks.append({"track_number": 1, "track_title": title_el.get_text(strip=True)})
        return tracks

    def _scrape_one(url):
        r = requests.get(url)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        tralbum = _parse_tralbum_json(soup)

        album_name = _get_album_title(soup, tralbum)
        artist = _get_artist(soup)

        rel_txt = _get_release_text(soup)
        release_date = _clean_date(rel_txt)
        release_year = _extract_release_year(release_date or rel_txt)

        label = _guess_label(soup)
        if not label:
            # Some pages include label-like owner in JSON (less consistent)
            if tralbum and tralbum.get("artist"):
                # If 'artist' differs from site/artist, leave as Self Release anyway
                pass
            label = "Self Release"

        # Tracks
        tracks = _tracks_from_tralbum(tralbum)
        if not tracks:
            tracks = _tracks_from_dom(soup)

        # Single detection: one track on page
        is_single = len(tracks) == 1

        rows = []
        for t in tracks:
            tn = t.get("track_number")
            title = t.get("track_title")
            is_remix, remixer, mix_type = _parse_mix_info(title or "")

            # If single, keep same columns; (requirement says: “if there is a single you have to give me
            # the columns remixer and mix type” — we already include them for all, so we’re good.)
            rows.append({
                "url": url,
                "album_name": album_name,
                "artist": artist,
                "label": label,
                "release_date": release_date,
                "release_year": release_year,
                "track_number": tn,
                "track_title": title,
                "is_remix": is_remix,
                "remixer": remixer,
                "mix_type": mix_type
            })
        return rows

    # ---------- dispatch ----------
    if isinstance(urls_or_df, pd.DataFrame):
        urls = urls_or_df[url_col].dropna().astype(str).tolist()
        out_rows = []
        total = len(urls)
        for i, u in enumerate(urls, start=1):
            _tqm_bar(i, total, "Bandcamp scrape")
            try:
                out_rows.extend(_scrape_one(u))
            except Exception as e:
                out_rows.append({
                    "url": u, "album_name": None, "artist": None, "label": "Self Release",
                    "release_date": None, "release_year": None,
                    "track_number": None, "track_title": None,
                    "is_remix": None, "remixer": None, "mix_type": None,
                    "_error": str(e)
                })
        df_tracks = pd.DataFrame(out_rows)
        # Merge back at album/URL level if you want per-row mapping; since tracks are multiple rows per URL,
        # we’ll just return the track-level DF (safer). You can merge/join upstream as needed.
        return df_tracks

    elif isinstance(urls_or_df, str):
        _tqm_bar(1, 1, "Bandcamp scrape")
        return pd.DataFrame(_scrape_one(urls_or_df))

    else:
        # assume iterable of strings
        urls = list(urls_or_df)
        out_rows = []
        total = len(urls)
        for i, u in enumerate(urls, start=1):
            _tqm_bar(i, total, "Bandcamp scrape")
            out_rows.extend(_scrape_one(u))
        return pd.DataFrame(out_rows)


In [6]:
# Example: single URL
df_bc = _bandcamp_2710_album_GET_df_tracks("https://brvss.bandcamp.com/album/editsitos-004")


TQM | Bandcamp scrape: 100%|████████████████████████████| 1/1


In [3]:
df_bc

,url,album_name,artist,label,release_date,release_year,track_number,track_title,is_remix,remixer,mix_type
0,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,1,la bebé (brvss jungleton mix),False,None,None
1,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,2,mes tas tentando (brvss club tool),False,None,None
2,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,3,rather lie (y2kid's eurodance mix),False,None,None
3,https://brvss.bandcamp.com/album/editsitos-004,editsitos 004,brvss,Self Release,2025-09-29,2025,4,soy mexicano (Pearl Jetta remix),True,Pearl Jetta,Remix
